## Local Inference on GPU
Model page: https://huggingface.co/timm/mobilenetv3_small_100.lamb_in1k

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/timm/mobilenetv3_small_100.lamb_in1k)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [ ]:
# @title
!pip install tensorflow tensorflow-datasets torch torchvision datasets

In [ ]:
import tensorflow_datasets as tfds
import tensorflow as tf
import timm
import pandas, numpy, matplotlib.pyplot
from datasets import load_dataset, DatasetDict
from PIL import Image

import torch
from torchvision import transforms
import torch.nn as nn
import torchvision.models as models
from torchvision.datasets import SUN397
from torch.utils.data import Subset
from torch.utils.data import DataLoader


In [ ]:
#using three pretrained models
model_mob = timm.create_model("hf_hub:timm/mobilenetv3_small_100.lamb_in1k", pretrained=True)
model_eff = timm.create_model("hf_hub:timm/efficientnet_b0.ra_in1k", pretrained=True)
model_res = timm.create_model("hf_hub:timm/resnet18.a1_in1k", pretrained=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/586 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/10.2M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/755 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

In [ ]:
#LOAD SUN397 DATABASE
#dataset = load_dataset("tanganke/sun397")  # downloads from Hugging Face
#print(dataset)

# Example: access one split
#example = dataset["train"][0]
#image = example["image"]
#label = example["label"]

#READ LABELS TO PICK SUBSET
#feature = dataset["train"].features["label"]
#print(feature.names)       # first 50 class names
#print(len(feature.names))       # should be 397

In [ ]:
#Classes I will be using
classes = [
    "beach",
    "bedroom",
    "kitchen",
    "living room",
    "office",
    "mountain",
    "highway",
    "street",
    "church indoor",
    "forest broadleaf",
]


In [ ]:
#Grab the subset of dataset to use
dataset = load_dataset("tanganke/sun397")  # downloads once and caches locally

# Inspect label names
id_to_label = dataset["train"].features["label"].int2str
label_to_id = dataset["train"].features["label"].str2int

class_ids = [label_to_id(name) for name in classes]

# Filter to keep only the chosen 10 classes
def is_easy_example(example):
    return example["label"] in class_ids

train_ds = dataset["train"].filter(is_easy_example)
val_ds   = dataset["validation"].filter(is_easy_example) if "validation" in dataset else dataset["test"].filter(is_easy_example)

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Filter:   0%|          | 0/19850 [00:00<?, ? examples/s]

Filter:   0%|          | 0/19850 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


In [ ]:
#SAVE SUBSET LOCALLY
subset = DatasetDict({
    "train": train_ds,
    "validation": val_ds,   # or "test" if that’s what you used
})

subset.save_to_disk("sun10_subset")

Saving the dataset (0/1 shards):   0%|          | 0/500 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = 10


In [ ]:
#TRANSFORM TRAINING AND VALIDATION DATA INTO
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

class HFDataset(torch.utils.data.Dataset):
    def __init__(self, hf_split, transform, easy_ids, id2label, label2id):
        self.ds = hf_split
        self.transform = transform
        self.easy_ids = set(easy_ids)
        # build compact 0–9 labels
        self.old2new = {old: new for new, old in enumerate(easy_ids)}

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        ex = self.ds[idx]
        img = ex["image"]
        if not isinstance(img, Image.Image):
            img = Image.fromarray(img)
        y_old = ex["label"]
        y = self.old2new[y_old]
        img = self.transform(img)
        return img, y

train_data = HFDataset(train_ds, train_transform, easy_class_ids, id2label, label2id)
val_data   = HFDataset(val_ds, val_transform, easy_class_ids, id2label, label2id)


train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_loader   = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def build_resnet18(num_classes=NUM_CLASSES, freeze_backbone=False):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    if freeze_backbone:
        for p in model.parameters():
            p.requires_grad = False
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model.to(device)

def build_efficientnet_b0(num_classes=NUM_CLASSES, freeze_backbone=False):
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    if freeze_backbone:
        for p in model.parameters():
            p.requires_grad = False
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model.to(device)

def build_mobilenet_v3(num_classes=NUM_CLASSES, freeze_backbone=False):
    model = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V1)
    if freeze_backbone:
        for p in model.parameters():
            p.requires_grad = False
    in_features = model.classifier[3].in_features
    model.classifier[3] = nn.Linear(in_features, num_classes)
    return model.to(device)

In [ ]:
def train_one_model(model, train_loader, val_loader, epochs=5, lr=1e-3):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)

    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * imgs.size(0)
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / total
        train_acc = correct / total

        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                _, preds = outputs.max(1)
                val_correct += preds.eq(labels).sum().item()
                val_total += labels.size(0)

        val_acc = val_correct / val_total
        print(f"Epoch {epoch+1}/{epochs} | loss={train_loss:.4f} | train_acc={train_acc:.3f} | val_acc={val_acc:.3f}")

    return model

In [ ]:
resnet18 = build_resnet18()
efficientnet_b0 = build_efficientnet_b0()
mobilenet_v3 = build_mobilenet_v3()

print("Training ResNet‑18")
resnet18 = train_one_model(resnet18, train_loader, val_loader, epochs=5, lr=1e-3)

print("Training EfficientNet‑B0")
efficientnet_b0 = train_one_model(efficientnet_b0, train_loader, val_loader, epochs=5, lr=1e-3)

print("Training MobileNetV3")
mobilenet_v3 = train_one_model(mobilenet_v3, train_loader, val_loader, epochs=5, lr=1e-3)